In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/project-king/sample_submission (3).csv
/kaggle/input/project-king/train.csv
/kaggle/input/project-king/test.csv
/kaggle/input/me-4127-e-project-2/sample_submission.csv
/kaggle/input/me-4127-e-project-2/train.csv
/kaggle/input/me-4127-e-project-2/test.csv


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from sklearn.metrics import log_loss
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

In [3]:
# Load datasets
train_data = pd.read_csv('/kaggle/input/me-4127-e-project-2/train.csv')
test_data = pd.read_csv('/kaggle/input/me-4127-e-project-2/test.csv')
sample_submission = pd.read_csv('/kaggle/input/me-4127-e-project-2/sample_submission.csv')

In [4]:
# Preprocess target
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_data['Status'])

# Drop target from features
X = train_data.drop(columns=['Status'])

In [5]:
# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()

In [6]:
# Define preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), numerical_cols),
        
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)
    ]
)

In [7]:
# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [8]:
# Define the base models for stacking
base_estimators = [
    ('log_reg', LogisticRegression(C=0.1, max_iter=2000, multi_class='multinomial', random_state=42)),
    ('xgb', xgb.XGBClassifier(objective='multi:softprob', num_class=3, eval_metric='mlogloss', 
                              learning_rate=0.2, max_depth=3, n_estimators=200, subsample=0.8, random_state=42))
]

In [9]:
# Define the meta-learner
meta_learner = LogisticRegression(max_iter=1000, random_state=42)


In [10]:
# Build the stacking classifier
stacking_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=5
)

In [11]:
# Create a pipeline with preprocessing and stacking model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', stacking_clf)
])

In [12]:
# Fit the model on training data
pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['id', 'N_Days', 'Age',
                                                   'Bilirubin', 'Cholesterol',
                                                   'Albumin', 'Copper',
                                                   'Alk_Phos', 'SGOT',
                                                   'Tryglicerides', 'Platelets',
                                                   'Prothrombin', 'Stage']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_freque...
                                                               interaction_constraints=None,
                                                               learning_rate=0.2,
                                                               max_bin=None,
                                                               max_cat_threshold=None,
                                                               max_cat_to_onehot=None,
                                                               max_delta_step=None,
                                                               max_depth=3,
                                                               max_leaves=None,
                                                               min_child_weight=None,
                                                               missing=nan,
                                                               monotone_constraints=None,
                                                               multi_strategy=None,
                                                               n_estimators=200,
                                                               n_jobs=None,
                                                               num_class=3,
                                                               num_parallel_tree=None, ...))],
                                    final_estimator=LogisticRegression(max_iter=1000,
                                                                       random_state=42)))])

In [13]:
# Evaluate with cross-validation log loss
stacking_cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_log_loss')
print("Cross-validation log loss for Stacking Ensemble:", -stacking_cv_scores.mean())


Cross-validation log loss for Stacking Ensemble: 0.3835659463215797


In [14]:
# Predict on validation set and calculate log loss
y_val_pred_stack = pipeline.predict_proba(X_val)
stacking_log_loss = log_loss(y_val, y_val_pred_stack)
print("Validation log loss for Stacking Ensemble:", stacking_log_loss)

Validation log loss for Stacking Ensemble: 0.3738379855255116


In [15]:
# Prepare test data and make predictions
X_test = test_data
y_test_pred_stack = pipeline.predict_proba(X_test)


In [16]:
# Prepare submission file
submission = pd.DataFrame(y_test_pred_stack, columns=label_encoder.inverse_transform([0, 1, 2]))
submission.insert(0, 'id', test_data['id'])
submission.columns = ['id', 'Status_C', 'Status_CL', 'Status_D']
submission.to_csv('submission.csv', index=False)
print("Stacking Ensemble submission file created successfully.")

Stacking Ensemble submission file created successfully.
